# 01 - Data Ingestion

Sync data from all external sources and load into the vector store.

**Steps:**
1. Sync participant profiles from Google Sheets
2. Fetch interview transcripts from Marvin
3. Fetch researcher notes from Google Docs
4. Load all data into ChromaDB vector store

In [ ]:
import sys, json, logging
from pathlib import Path

# Add project root to path
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s: %(message)s')
logger = logging.getLogger(__name__)

from config.settings import PROCESSED_DIR, REFERENCE_DIR

## Step 1: Sync Marvin Transcripts

Pull interview transcripts from Marvin MCP and cache locally.

In [ ]:
from src.loaders.marvin_loader import MarvinLoader

loader = MarvinLoader()

# List available interviews
interviews = loader.list_interviews()
print(f"Found {len(interviews)} interviews in Marvin")
for i in interviews[:5]:
    print(f"  - {i.get('name', '')} (ID: {i.get('id', '')})")

In [ ]:
# Sync all available transcripts
results = loader.sync_all_interviews()
print(f"\nSync complete:")
print(f"  Transcripts cached: {results.get('transcripts_cached', 0)}")
print(f"  Summaries cached: {results.get('summaries_cached', 0)}")

## Step 2: Load Into Vector Store

In [ ]:
from src.storage.vector_store import VectorStore

vs = VectorStore()

# Load participant map
with open(REFERENCE_DIR / 'participant_map.json') as f:
    pmap = json.load(f)

# Load cached transcripts
transcript_dir = PROCESSED_DIR / 'transcripts'
loaded = 0
if transcript_dir.exists():
    for path in transcript_dir.glob('*.json'):
        with open(path) as f:
            data = json.load(f)
        text = data.get('transcript_text', '')
        if not text:
            continue
        file_id = data.get('file_id', '')
        vs.add_document(
            doc_id=f'transcript_{file_id}',
            text=text,
            metadata={'source': 'interview', 'file_id': str(file_id)},
        )
        loaded += 1

print(f'Loaded {loaded} transcripts into vector store ({vs.count} total chunks)')

## Step 3: Verify Ingestion

In [ ]:
# Test search
results = vs.search('food delivery ordering habits', n_results=3)
print(f'Search returned {len(results)} results:')
for r in results:
    print(f"  - [{r['metadata'].get('file_id', '')}] {r['text'][:100]}...")